# End-to-End SLM Pipeline: Synthetic Data, Training & Inference
This notebook covers the full pipeline: generating synthetic training data using DeepSeek via OpenRouter, fine-tuning an SLM (Gemma 4 E2B) for PII Redaction, testing the trained model in-memory, and exporting it for edge deployment on your Jetson Orin Nano.

In [1]:
%%capture
!pip install requests unsloth
!pip install --force-reinstall "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

### 1. Synthetic Data Generation (OpenRouter & DeepSeek Pro)
Run this cell to generate your JSONL dataset using your OpenRouter API key.

In [1]:
import requests
import json
import getpass
from google.colab import userdata

# Retrieve the secret API key stored in Colab Secrets
try:
    api_key = userdata.get('OPENROUTER_API_KEY')
    print("Successfully loaded OPENROUTER_API_KEY from Colab Secrets.")
except Exception as e:
    raise ValueError("Could not find 'OPENROUTER_API_KEY' in Colab Secrets. Please check your Secrets sidebar.") from e

system_prompt = """
You are an expert data generator for a Spanish LegalTech NLP pipeline.
Generate 100 highly diverse training examples of Spanish taxpayers or companies asking questions about real estate, taxes, or corporate law.

Output strictly in a JSON array format. Each object must have:
1. "instruction": "Redact all PII (organizations, people, locations, financial amounts, and IDs) from the following legal query and provide the mapping."
2. "input": A realistic legal query containing fake PII.
3. "output": The Sanitized string using tags like [ORG_1], [PER_1], [LOC_1], [MONEY_1], [ID_1], followed by '\nMapping: ' and the tag relationships.
"""

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

payload = {
    "model": "deepseek/deepseek-chat", # DeepSeek V3/V4 API endpoint
    "messages": [{"role": "system", "content": system_prompt}],
    "response_format": {"type": "json_object"}
}

print("Calling OpenRouter API...")
response = requests.post("https://openrouter.ai/api/v1/chat/completions", headers=headers, json=payload)

if response.status_code == 200:
    content = response.json()["choices"][0]["message"]["content"]
    data = json.loads(content)
    dataset = data if isinstance(data, list) else list(data.values())[0]

    with open("legal_redaction_dataset.jsonl", "w", encoding="utf-8") as f:
        for item in dataset:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    print(f"Successfully saved {len(dataset)} examples to legal_redaction_dataset.jsonl")
else:
    print(f"API Error {response.status_code}: {response.text}")

Successfully loaded OPENROUTER_API_KEY from Colab Secrets.
Calling OpenRouter API...
Successfully saved 66 examples to legal_redaction_dataset.jsonl


### 2. Load Model & Setup Training

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2-2b-it-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1110 23:33:32.820000 6087 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1110 23:33:32.870000 6087 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.3: Fast Gemma2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Unsloth: Will load unsloth/gemma-2-2b-it-bnb-4bit as a legacy tokenizer.
Unsloth 2026.11.3 patched 26 layers with 26 QKV layers, 26 O layers and 26 MLP layers.


Unsloth: compiling each transformer block as one region (Gemma2DecoderLayer).


In [3]:
from datasets import load_dataset

prompt_template = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    texts = []
    for instruction, input_text, output in zip(examples["instruction"], examples["input"], examples["output"]):
        text = prompt_template.format(instruction=instruction, input=input_text, output=output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts }

dataset = load_dataset("json", data_files="legal_redaction_dataset.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/66 [00:00<?, ? examples/s]

In [4]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 200, # Change to 200-300 for final training
        learning_rate = 2e-4,
        fp16 = True,
        bf16 = False,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/66 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 66 | Num Epochs = 23 | Total steps = 200
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 20,766,720 of 2,635,108,608 (0.79% trained)


Step,Training Loss
1,2.223099
2,2.251875
3,2.159567
4,1.849606
5,1.681917
6,1.247490
7,1.006917
8,0.685587
9,0.446418
10,0.336997


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-200/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-200.


### 3. Test Inference (Local Verification)
Let's test the newly trained LoRA weights right here in the notebook before exporting.

In [5]:
# Enable 2x faster inference natively in Unsloth
FastLanguageModel.for_inference(model)

test_input = "La empresa Innovaciones Digitales SL, con NIF B98765432, compró un servidor en Madrid por 15.000€."

inputs = tokenizer(
[
    prompt_template.format(
        instruction="Redact all PII (organizations, people, locations, financial amounts, and IDs) from the following legal query and provide the mapping.",
        input=test_input,
        output="", # Leave output blank for generation
    )
], return_tensors = "pt").to("cuda")

print("\n--- GENERATING RESPONSE ---\n")
outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

# Print only the generated portion (splitting at the Response tag)
print(response.split("### Response:\n")[1])

Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- GENERATING RESPONSE ---

La empresa [ORG_1] con [ORG_2] compró un servidor en [LOC_1] por [MONEY_1].
Mapping: [ORG_1] = Innovaciones Digitales SL, [ORG_2] = B98765432, [LOC_1] = Madrid, [MONEY_1] = 15.000€.


### 4. Export for Edge Deployment

In [7]:
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

# Export to GGUF (Q4_K_M is the standard format for Ollama/Jetson deployment)
model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
print("Export complete! You can now download the .gguf file from the files panel.")

Unsloth: Restored added_tokens_decoder metadata in lora_model/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in lora_model.
Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/913 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in model/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in model.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 5.23GB            

model.safetensors: downloading bytes:           |  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [02:00<00:00, 120.32s/it]


Unsloth: Merge process complete. Saved to `/content/model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b12246-mix-7c77012 (app-b12246-mix-7c77012-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['model_gguf/gemma-2-2b-it.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...


KeyboardInterrupt: 

### 5. 3-Way inference comparison: fined tuned model vs raw OS model vs commercial LLM

In [11]:
import json
import requests
from google.colab import userdata
from unsloth import FastLanguageModel

# -------------------------------------------------------------
# 1. SETUP & TEST DATASET (4 DIVERSE LEGAL CASES)
# -------------------------------------------------------------
try:
    api_key = userdata.get('OPENROUTER_API_KEY')
except Exception as e:
    raise ValueError("Please ensure 'OPENROUTER_API_KEY' is set in Colab Secrets.") from e

instruction_text = "Redact all PII (organizations, people, locations, financial amounts, and IDs) from the following legal query and provide the mapping."

eval_queries = [
    {
        "domain": "Real Estate / Asset Sale",
        "input": "La empresa Innovaciones Digitales SL, con NIF B98765432, compró un servidor en Madrid por 15.000€ a través del apoderado Carlos Ruíz."
    },
    {
        "domain": "Inheritance Tax / Personal Tax",
        "input": "Mi madre, Maria del Carmen Gómez (DNI 12345678A), quiere donarme un inmueble en Calle Balmes 45, Barcelona valorado en 280.000€."
    },
    {
        "domain": "Corporate Restructuring / Mergers",
        "input": "La sociedad Grupo Logístico del Norte SA (CIF A55443322) ha firmado una fusión con Transportes Ibéricos SL en Valencia por un importe de 1.200.000€."
    },
    {
        "domain": "Labor Law / Severance Pay",
        "input": "El empleado Antonio Fernández ha finalizado su contrato con la firma Consultoría Tech SL recibiendo una indemnización de 42.500€ liquidados en Sevilla."
    }
]

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

# -------------------------------------------------------------
# 2. RUN 3-WAY BENCHMARK LOOP
# -------------------------------------------------------------
print("🚀 Starting 3-Way Benchmark Execution Across 4 Scenarios...\n")

results = []

for idx, scenario in enumerate(eval_queries, 1):
    test_input = scenario["input"]
    domain = scenario["domain"]

    formatted_prompt = prompt_template.format(
        instruction=instruction_text,
        input=test_input,
        output=""
    )

    # A) Commercial LLM (DeepSeek via OpenRouter)
    payload = {
        "model": "deepseek/deepseek-chat",
        "messages": [{"role": "user", "content": f"{instruction_text}\n\nInput: {test_input}"}],
        "temperature": 0.0
    }
    response = requests.post("https://openrouter.ai/api/v1/chat/completions", headers=headers, json=payload)
    deepseek_out = response.json()["choices"][0]["message"]["content"] if response.status_code == 200 else f"Error: {response.text}"

    # B) Raw Base Model (Untrained Gemma 2B)
    with model.disable_adapter():
        FastLanguageModel.for_inference(model)
        inputs_raw = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")
        outputs_raw = model.generate(**inputs_raw, max_new_tokens=256, use_cache=True)
        raw_full = tokenizer.batch_decode(outputs_raw, skip_special_tokens=True)[0]
        raw_out = raw_full.split("### Response:\n")[-1].strip()

    # C) Fine-Tuned SLM (Gemma 2B + LoRA)
    FastLanguageModel.for_inference(model)
    inputs_ft = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")
    outputs_ft = model.generate(**inputs_ft, max_new_tokens=256, use_cache=True)
    ft_full = tokenizer.batch_decode(outputs_ft, skip_special_tokens=True)[0]
    ft_out = ft_full.split("### Response:\n")[-1].strip()

    results.append({
        "id": idx,
        "domain": domain,
        "input": test_input,
        "deepseek": deepseek_out,
        "raw": raw_out,
        "fine_tuned": ft_out
    })

# -------------------------------------------------------------
# 3. STRUCTURED COMPARISON DISPLAY
# -------------------------------------------------------------
print("\n" + "█"*90)
print("  SLM PII REDACTION PIPELINE: 3-WAY BENCHMARK EVALUATION")
print("█"*90 + "\n")

for item in results:
    print(f"📌 SCENARIO {item['id']}/4: {item['domain'].upper()}")
    print(f"📥 INPUT: \"{item['input']}\"\n")

    print("┌" + "─"*88)
    print("│ 1️⃣ COMMERCIAL LLM (DeepSeek Chat - General Prompt)")
    print("├" + "─"*88)
    for line in item['deepseek'].split('\n'):
        print(f"│  {line}")
    print("└" + "─"*88 + "\n")

    print("┌" + "─"*88)
    print("│ 2️⃣ RAW BASE SLM (Gemma 2B - Untrained)")
    print("├" + "─"*88)
    for line in item['raw'].split('\n'):
        print(f"│  {line}")
    print("└" + "─"*88 + "\n")

    print("┌" + "─"*88)
    print("│ 3️⃣ FINE-TUNED SLM (Gemma 2B + LoRA Adapter)")
    print("├" + "─"*88)
    for line in item['fine_tuned'].split('\n'):
        print(f"│  {line}")
    print("└" + "─"*88)

    print("\n" + "═"*90 + "\n")

🚀 Starting 3-Way Benchmark Execution Across 4 Scenarios...



Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati


██████████████████████████████████████████████████████████████████████████████████████████
  SLM PII REDACTION PIPELINE: 3-WAY BENCHMARK EVALUATION
██████████████████████████████████████████████████████████████████████████████████████████

📌 SCENARIO 1/4: REAL ESTATE / ASSET SALE
📥 INPUT: "La empresa Innovaciones Digitales SL, con NIF B98765432, compró un servidor en Madrid por 15.000€ a través del apoderado Carlos Ruíz."

┌────────────────────────────────────────────────────────────────────────────────────────
│ 1️⃣ COMMERCIAL LLM (DeepSeek Chat - General Prompt)
├────────────────────────────────────────────────────────────────────────────────────────
│  **Redacted Text:**  
│  La empresa [ORGANIZATION_1], con NIF [ID_1], compró un servidor en [LOCATION_1] por [AMOUNT_1]€ a través del apoderado [PERSON_1].  
│  
│  **Mapping:**  
│  - [ORGANIZATION_1]: Innovaciones Digitales SL  
│  - [ID_1]: B98765432  
│  - [LOCATION_1]: Madrid  
│  - [AMOUNT_1]: 15.000  
│  - [PERSON_1]: Carlos Ru

### 6. Export and save all session materials to Drive


In [8]:
import os
import shutil
from google.colab import drive

# 1. Montar Google Drive
drive.mount('/content/drive')

# 2. Definir la carpeta de destino en Drive usando el ID de la URL
# ID: YOUR_GOOGLE_DRIVE_FOLDER_ID
target_folder_id = "YOUR_GOOGLE_DRIVE_FOLDER_ID"

# Buscar la ruta física asignada por Drive en la sesión montada
destination_dir = None
drive_root = "/content/drive/MyDrive"

# Búsqueda o fallback directo si la carpeta existe en la raíz
for root, dirs, files in os.walk("/content/drive"):
    if target_folder_id in root:
        destination_dir = root
        break

# Si no localiza la ID directamente por estructura interna de FUSE,
# se crea/garantiza un directorio directo para el proyecto:
if not destination_dir:
    destination_dir = os.path.join(drive_root, "SLM_Legal_Redaction_Exports")
    os.makedirs(destination_dir, exist_ok=True)

print(f"📁 Exportando artefactos a: {destination_dir}")

# 3. Lista de artefactos a copiar
# - legal_redaction_dataset.jsonl (para el repo de GitHub)
# - lora_model/ (pesos LoRA ligeros para GitHub / Hugging Face)
# - modelo .gguf (pesos completos unificados para Ollama / despliegue Edge)
artifacts = ["legal_redaction_dataset.jsonl", "lora_model"]

for f in os.listdir("."):
    if f.endswith(".gguf"):
        artifacts.append(f)

for item in artifacts:
    if os.path.exists(item):
        dest_path = os.path.join(destination_dir, item)
        if os.path.isfile(item):
            shutil.copy(item, dest_path)
            print(f"  ✅ Archivo copiado: {item}")
        elif os.path.isdir(item):
            if os.path.exists(dest_path):
                shutil.rmtree(dest_path)
            shutil.copytree(item, dest_path)
            print(f"  ✅ Carpeta copiada: {item}/")
    else:
        print(f"  ⚠️ No encontrado en el entorno local de Colab: {item}")

print("\n🚀 ¡Exportación completada! Todos tus archivos y pesos están listos en tu carpeta de Google Drive.")

Mounted at /content/drive
📁 Exportando artefactos a: /content/drive/MyDrive/SLM_Legal_Redaction_Exports
  ✅ Archivo copiado: legal_redaction_dataset.jsonl
  ✅ Carpeta copiada: lora_model/

🚀 ¡Exportación completada! Todos tus archivos y pesos están listos en tu carpeta de Google Drive.
